# Gradio Day!

Today we will build User Interfaces using the outrageously simple Gradio framework.

Prepare for joy!

Please note: your Gradio screens may appear in 'dark mode' or 'light mode' depending on your computer settings.

In [ ]:
import os 
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, update_display, Markdown

In [ ]:
import gradio as gr

In [ ]:
# Loading the keys
load_dotenv(override=True)
gemini_api_key = os.getenv('GOOGLE_API_KEY')
open_router_api_key = os.getenv('OPENROUTER_API_KEY')

if gemini_api_key and gemini_api_key.startswith('AIz'):
    print(f'Google API key found and looks good so far')
else:
    print(f'Google API key not found')

if open_router_api_key and open_router_api_key.startswith('sk-or'):
    print('Open Router API key found and good so far')
else:
    print('Api Key not found')


In [ ]:
# Defining base urls, and LLM objects # Connect to OpenAI client library

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
open_router_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
open_router = OpenAI(api_key=open_router_api_key, base_url=open_router_url)
ollama = OpenAI(api_key='ollama', base_url=ollama_url)

gemini_model_gemma_4_31b = 'google/gemma-4-31b-it:free'
gpt_oss_model = 'openai/gpt-oss-120b:free'
inclusionai_model_ring_2_1t = 'inclusionai/ring-2.6-1t:free'
baidu_model = "baidu/cobuddy:free"
deepseek_v4_flash_model = "deepseek/deepseek-v4-flash:free"


In [ ]:
def call_model(client_llm, model_name, messages, **kwargs):
    response = client_llm.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content


In [ ]:
system_prompt = 'You are an helpful assistant'

In [ ]:
messages = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": "What is today's date"}]

open_router_response = call_model(open_router, gpt_oss_model, messages=messages)
display(Markdown(open_router_response))

In [ ]:
messages = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": "What is today's date"}]

gemini_response = call_model(gemini, 'gemini-2.5-flash-lite', messages=messages)
display(Markdown(gemini_response))

## User Interface time!

In [ ]:
# Defining basic shout function

def shout(text):
    print(f'Shout has been called with the text {text}')
    return text.upper()


In [ ]:
demo = gr.Interface(fn=shout, inputs='textbox', outputs='textbox', flagging_mode='never')
demo.launch()

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">NOTE: Using Gradio's Share tool</h2>
            <span style="color:#900;">I'm about to show you a really cool way to share your Gradio UI with others. This deploys your gradio app as a demo on gradio's website, and then allows gradio to call the 'shout' function. This uses an advanced technology known as 'HTTP tunneling' (like ngrok for people who know it) which isn't allowed by many Antivirus programs and corporate environments. If you get an error, just skip the next cell.<br/>
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Adding share=True means that it can be accessed publically
# A more permanent hosting is available using a platform called Spaces from HuggingFace, which we will touch on next week
# NOTE: Some Anti-virus software and Corporate Firewalls might not like you using share=True. 
# If you're at work on on a work network, I suggest skip this test.

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [ ]:
# Adding inbrowser=True opens up a new browser window automatically

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

## Adding authentication

Gradio makes it very easy to have userids and passwords

Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env

In [ ]:
load_dotenv(override=True)

username = os.getenv('USER_NAME')
password = os.getenv('PASSWORD')


# Forcing Dark Mode

In [ ]:
# Define this variable and then pass js=force_dark_mode when creating the Interface

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

### Rewriting Gradio code in structured format

In [ ]:
input_messages = gr.Textbox(label='Your message:', placeholder='Input Text', info='Enter a message to shout: ', lines=7)
output_messages = gr.Textbox(label='Response: ', placeholder='Output Text', info='Output Message', lines=8)

app = gr.Interface(
    fn=shout,
    title="Shout",
    inputs=[input_messages],
    outputs=[output_messages],
    flagging_mode='never',
    examples=['hello', 'hola']
)

app.launch()


# Calling LLM using Gradio UI

### 1. GPT OSS model

In [ ]:
## GPT endpoint

def message_gpt_oss(message):
    messages = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": message}]
    response = call_model(open_router, gpt_oss_model, messages=messages)
    return response

In [ ]:
# Calling GPT from UI

input_messages = gr.Textbox(label='Your message:', info='Enter a message for GPT-oss-model', lines=7)
output_messages = gr.Textbox(label='Response:', lines=8)

app = gr.Interface(
    fn=message_gpt_oss,
    title='Chat LLM',
    inputs=[input_messages],
    outputs=[output_messages],
    examples=['Explain transformers to a laymen', 'Explain transformers to an aspiring AI engineer'],
    flagging_mode='never'
)

app.launch()

In [ ]:
# Using Markdown

system_message = "You are a helpful assistant that responds in markdown without code blocks"

input_messages = gr.Textbox(label='Your message:', info='Enter a message for GPT-oss-model', lines=7)
output_messages = gr.Markdown('Response:')

app = gr.Interface(
    fn=message_gpt_oss,
    title='GPT',
    inputs=[input_messages],
    outputs=[output_messages],
    examples=['Explain the Transformer architecture to a layperson',
              'Explain the Transformer architecture to an aspiring AI engineer'],
    flagging_mode='never'
)

app.launch()


In [ ]:
# Streaming the results

def stream_gpt_oss(message):
    streaming = open_router.chat.completions.create(
        model=gpt_oss_model,
        messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": message}],
        stream=True
    )
    result = ''
    for chunk in streaming:
        result += chunk.choices[0].delta.content or '' 
        yield result


In [ ]:
# yield keyword

message_input = gr.Textbox(label='Your message:', info='Enter a message for GPT-oss-model', lines=7)
message_output = gr.Markdown('Response:')

app = gr.Interface(
    title='GPT-OSS',
    fn=stream_gpt_oss,
    inputs=[message_input],
    outputs=[message_output],
    examples=["Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer"],
    flagging_mode='never'
)

app.launch()


### 2. Gemini 3.1-flash lite model

In [ ]:
## Stream gemini 3.1 - flash lite, 

def stream_gemini_model(message):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': message}
    ]
    streaming = gemini.chat.completions.create(
        model='gemini-3.1-flash-lite',
        messages=messages,
        stream=True
    ) 
    result = '' 
    for chunk in streaming:
        result += chunk.choices[0].delta.content or '' 
        yield result


In [ ]:
message_input = gr.Textbox(label='Your message:', info='Enter a message for Gemini 3.5 Flash Lite', lines=7)
message_output = gr.Markdown('Response:')

app = gr.Interface(
    fn=stream_gemini_model,
    title='Gemini Model',
    inputs=[message_input],
    outputs=[message_output],
    examples=["Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer"],
    flagging_mode='never'
)

app.launch()


## And now getting fancy

Remember to check the Intermediate Python Guide if you're unsure about generators and "yield"

In [ ]:
## Drop down for both models 

def select_model(prompt, model):
    if (model=='GPT-OSS'):
        result = stream_gpt_oss(prompt)
    elif (model=='Gemini'):
        result = stream_gemini_model(prompt)
    else:
        raise 'ValueError' 
    yield from result


In [ ]:
message_input = gr.Textbox(label='Enter your message:', info='Enter a message for the LLM', lines=7)
message_dropdown = gr.Dropdown(['GPT-OSS', 'Gemini'], label='Select model', value='GPT-OSS')
message_output = gr.Markdown('Response:')

app = gr.Interface(
    fn=select_model,
    title='LLMs',
    inputs=[message_input, message_dropdown],
    outputs=[message_output],
    examples=[
        ["Explain the Transformer architecture to a layperson", 'GPT-OSS'],
        ["Explain the Transformer architecture to an aspiring AI engineer", 'Gemini']],
    flagging_mode='never'
)

app.launch()


# Building a company brochure generator

Now you know how - it's simple!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you read the next few cells</h2>
            <span style="color:#900;">
                Try to do this yourself - go back to the company brochure in week1, day5 and add a Gradio UI to the end. Then come and look at the solution.
            </span>
        </td>
    </tr>
</table>